# Web Agent Action Prediction — DPO Pipeline (A100 80GB)

**실행 순서**
```
0. 환경 설정     GPU 확인 · 패키지 설치
1. 경로 설정     파일 업로드(ZIP) & 경로 검증
2. DPO 데이터    build_dpo_dataset.py  →  data/dpo_dataset.json
3. DPO 학습      train_dpo.py          →  lora_model_dpo/
4. 라우팅 추론   inference_lora_router.py  →  submission_lora_router.csv
5. 결과 검증     op 분포 · fallback 비율 · 다운로드
```

**A100 80GB 최적화 요약**

| 항목 | 값 | 근거 |
|------|-----|------|
| 모델 정밀도 | bfloat16 (4bit 불필요) | 14B × 2bytes = 28GB |
| 학습 배치 | 4 (effective 16) | 28GB + LoRA 그래디언트 여유 충분 |
| 추론 배치 | 128 | SFT+DPO 동시 로드(56GB) 후 잔여 24GB 활용 |
| 모델 로딩 전략 | SFT+DPO 동시 로드 | 56GB < 80GB → Pass 간 재로드 불필요 |

---
## 0. 환경 설정

In [ ]:
import subprocess, torch

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
print(result.stdout)

assert torch.cuda.is_available(), "GPU 없음 — 런타임 유형 변경 → A100 선택"
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"PyTorch CUDA : {torch.version.cuda}")
print(f"VRAM         : {vram:.1f} GB")
assert vram >= 70, f"A100 80GB 필요. 현재: {vram:.1f} GB"
print("\n✅ A100 80GB 확인 완료")

In [ ]:
import importlib

if importlib.util.find_spec('unsloth') is None:
    print("Installing packages...")
    !pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --quiet "trl>=0.8.6" "peft>=0.10.0" "datasets>=2.18.0"
    print("✅ 설치 완료 — 런타임 재시작 후 이 셀부터 다시 실행하세요")
    import os; os.kill(os.getpid(), 9)
else:
    import unsloth, trl, peft, datasets
    print(f"unsloth : {unsloth.__version__}")
    print(f"trl     : {trl.__version__}")
    print(f"peft    : {peft.__version__}")
    print(f"datasets: {datasets.__version__}")
    print("\n✅ 패키지 확인 완료")

---
## 1. 파일 업로드 및 경로 설정

로컬에서 압축한 `my_code_0514from0508.zip` (data 폴더 및 lora_model 포함) 파일을 왼쪽 폴더 탭을 이용해 업로드하거나 아래 코드로 압축을 풉니다.

In [ ]:
# 파일 수동 업로드 (my_code_0514from0508.zip) 후 아래 명령어 주석을 풀고 실행
# !unzip -q -o /content/my_code_0514from0508.zip -d /content/

In [ ]:
import os, sys

# ─── 이 경로만 환경에 맞게 수정 ───────────────────────────────────
BASE_DIR = "/content/my_code_0514from0508"
# ──────────────────────────────────────────────────────────────────

DATA_DIR     = os.path.join(os.path.dirname(BASE_DIR), "data")
LORA_SFT_DIR = os.path.join(BASE_DIR, "lora_model")
LORA_DPO_DIR = os.path.join(BASE_DIR, "lora_model_dpo")
DPO_DATA     = os.path.join(DATA_DIR, "dpo_dataset.json")
TRAIN_CSV    = os.path.join(DATA_DIR, "train.csv")
TEST_CSV     = os.path.join(DATA_DIR, "test.csv")
SUBMISSION   = os.path.join(BASE_DIR, "submission_lora_router.csv")
LOG_TRAIN    = os.path.join(BASE_DIR, "train_dpo.log")
LOG_INFER    = os.path.join(BASE_DIR, "inference_router.log")

sys.path.insert(0, BASE_DIR)

for name, path in [
    ("BASE_DIR",    BASE_DIR),
    ("DATA_DIR",    DATA_DIR),
    ("train.csv",   TRAIN_CSV),
    ("test.csv",    TEST_CSV),
    ("lora_model/", LORA_SFT_DIR),
]:
    ok = os.path.exists(path)
    print(f"{'✅' if ok else '❌'} {name:14s}: {path}")

---
## 2. DPO 데이터셋 생성

`build_dpo_dataset.py` → `data/dpo_dataset.json`

- 필터: `real_web` 중 gold attrs 있는 행만
- Hard Negative: 전 우선순위 similarity 기반 (random 없음)
- chosen / rejected: 각각 다른 contrastive reasoning
- 예상 출력: ~2,979쌍 / 소요 시간: 2~5분 (CPU)

In [ ]:
import json

if os.path.exists(DPO_DATA):
    with open(DPO_DATA, 'r', encoding='utf-8') as f:
        existing = json.load(f)
    print(f"기존 dpo_dataset.json: {len(existing)}쌍")
    if existing:
        s = existing[0]
        same = s['chosen'].split('</think>')[0] == s['rejected'].split('</think>')[0]
        if same:
            print("⚠️  이전 버전 파일 — 아래 셀을 실행해 재생성하세요 (contrastive reasoning 미적용)")
        else:
            print("✅ contrastive reasoning 적용 확인 — 재생성 불필요")
else:
    print("dpo_dataset.json 없음 → 아래 셀 실행")

In [ ]:
!python "{BASE_DIR}/build_dpo_dataset.py"

In [ ]:
assert os.path.exists(DPO_DATA)
with open(DPO_DATA, 'r', encoding='utf-8') as f:
    pairs = json.load(f)

print(f"총 DPO 쌍: {len(pairs)}")
for key in ('prompt', 'chosen', 'rejected'):
    assert key in pairs[0], f"'{key}' 필드 누락"

same = sum(1 for p in pairs
           if p['chosen'].split('</think>')[0] == p['rejected'].split('</think>')[0])
print(f"reasoning 동일 쌍: {same} / {len(pairs)}")
assert same == 0, "contrastive reasoning 미적용"
print("\n✅ DPO 데이터셋 검증 완료")

---
## 3. DPO 학습

`train_dpo.py` → `lora_model_dpo/`

| 항목 | 값 |
|------|----|
| SFT 처리 | `merge_and_unload()` → reference = SFT 보장 |
| DPO LoRA | r=32, alpha=64 |
| beta / lr | 0.1 / 5e-5 |
| 배치 | per_device=4, grad_accum=4 (effective=16) |
| 예상 시간 | A100 기준 약 15~25분 |

In [ ]:
import torch, gc
gc.collect(); torch.cuda.empty_cache()
free  = torch.cuda.mem_get_info()[0] / 1024**3
total = torch.cuda.mem_get_info()[1] / 1024**3
print(f"학습 전 VRAM : {free:.1f} / {total:.1f} GB")
print(f"예상 사용량  : 약 40~50 GB (14B bfloat16 + LoRA 그래디언트 + 배치 4)")

In [ ]:
!python "{BASE_DIR}/train_dpo.py" 2>&1 | tee "{LOG_TRAIN}"

In [ ]:
import glob

assert os.path.exists(LORA_DPO_DIR), f"lora_model_dpo/ 없음: {LORA_DPO_DIR}"
files = [f for f in glob.glob(os.path.join(LORA_DPO_DIR, '**', '*'), recursive=True)
         if os.path.isfile(f)]
print(f"파일 수: {len(files)}")
for f in sorted(files):
    mb = os.path.getsize(f) / 1024**2
    print(f"  {os.path.relpath(f, LORA_DPO_DIR):45s} {mb:7.1f} MB")

cfg_path = os.path.join(LORA_DPO_DIR, 'adapter_config.json')
if os.path.exists(cfg_path):
    with open(cfg_path) as f:
        cfg = json.load(f)
    print(f"\nadapter r={cfg.get('r')}, alpha={cfg.get('lora_alpha')}")
print("\n✅ DPO 모델 검증 완료")

In [ ]:
import re

with open(LOG_TRAIN, 'r') as f:
    log = f.read()

losses = [float(x) for x in re.findall(r"'loss':\s*([0-9.]+)", log)]
if losses:
    print(f"Loss 기록 수 : {len(losses)}")
    print(f"초기 loss   : {losses[0]:.4f}")
    print(f"최종 loss   : {losses[-1]:.4f}")
    status = '✅ 감소' if losses[-1] < losses[0] else '⚠️  미감소 (lr 또는 beta 조정 권장)'
    print(f"감소 여부   : {status}")
else:
    print("loss 로그를 찾지 못했습니다")

---
## 4. 라우팅 추론

`inference_lora_router.py` → `submission_lora_router.csv`

| Pass | 대상 | 모델 | 비고 |
|------|------|------|------|
| Pass 1 | workflow + real_web (attrs 없음) | SFT | |
| Pass 2 | real_web (attrs 있음) | DPO | 재로드 없이 즉시 추론 |

**A100 80GB**: SFT(28GB) + DPO(28GB) = 56GB 동시 로드 →  
Pass 전환 시 모델 교체 없음, 배치 128로 속도 최대화, 예상 시간 약 30~40분

In [ ]:
import torch, gc, pandas as pd
gc.collect(); torch.cuda.empty_cache()
free  = torch.cuda.mem_get_info()[0] / 1024**3
total = torch.cuda.mem_get_info()[1] / 1024**3
print(f"추론 전 VRAM : {free:.1f} / {total:.1f} GB")
print(f"SFT+DPO 동시 로드 예정: ~56 GB")
print(f"배치 버퍼 여유: ~{total - 56:.1f} GB")
print(f"test.csv 행 수: {len(pd.read_csv(TEST_CSV))}")

In [ ]:
!python "{BASE_DIR}/inference_lora_router.py" 2>&1 | tee "{LOG_INFER}"

---
## 5. 결과 검증

In [ ]:
import pandas as pd

assert os.path.exists(SUBMISSION), f"submission 없음: {SUBMISSION}"
sub = pd.read_csv(SUBMISSION)

print(f"행 수   : {len(sub)}")
print(f"컬럼    : {list(sub.columns)}")
print(f"\n[op 분포]")
print(sub['op'].value_counts().to_string())

null_cnt = sub.isnull().sum().sum()
print(f"\nnull    : {null_cnt}건 {'✅' if null_cnt == 0 else '⚠️'}")

fb = (sub['target_id'].astype(str) == '0').sum()
fb_pct = fb / len(sub) * 100
print(f"fallback: {fb}건 ({fb_pct:.1f}%) {'✅' if fb_pct < 5 else '⚠️  5% 초과'}")

sub.head(10)

In [ ]:
with open(LOG_INFER, 'r') as f:
    log = f.read()

for line in log.split('\n'):
    if any(k in line for k in ['라우팅', 'VRAM', 'Fallback', '완료', 'Pass']):
        print(line)

In [ ]:
try:
    from google.colab import files
    files.download(SUBMISSION)
    print(f"✅ 다운로드: {os.path.basename(SUBMISSION)}")
except ImportError:
    print(f"파일 위치: {SUBMISSION}")

---
## 부록: 트러블슈팅

### OOM — 추론 배치 축소
`inference_lora_router.py` 상단 `BATCH_SIZE = 128` → `64`

### DPO loss NaN
`train_dpo.py`에서 `LEARNING_RATE = 5e-5` → `2e-5` 또는 `DPO_BETA = 0.1` → `0.3`

### lora_model/ 없음
SFT 학습을 먼저 실행:  `python my_code_0514from0508/train.py`